# worldmap 渲染引擎的部分逻辑

编辑器仅可直接操作 worldmap.json 中的数据，引擎内部按以下逻辑实时计算最终渲染位置

---

## 1. 统一坐标缩放

$$X_{w}=\texttt{m\_position.x} \cdot \frac{\text{resolution}}{600}$$
$$Y_{w}=\texttt{m\_position.y} \cdot \frac{\text{resolution}}{600}$$
$$\text{resolution} \in \{384,640,768,1536\}$$

## 2.mapPieces 结点仿射变换

### 2.1图像
- 初始原点为左上角，坐标系为屏幕坐标，x轴向右，y轴向下，这会改变旋转方向
- 右侧变换先生效，先生效的变换只影响对象本身而不会改变后续变换的坐标系

$$ \theta=- \texttt{m\_rotationAngle} \times \frac{\pi}{180}$$

$$
\mathbf{T}_{I} =
\underbrace{
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
}_{(6)\text{平移至世界锚点}}
\times
\underbrace{
\begin{pmatrix}
\text{scale}_{x} & 0 & 0 \\
0 & \text{scale}_{y} & 0 \\
0 & 0 & 1
\end{pmatrix}
}_{(5)\text{自由缩放，不一定等比例}}
\times
\underbrace{
\begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & \dfrac{\text{height}}{2} \\
0 & 0 & 1
\end{pmatrix}\\
}_{(4)\text{向下平移半高}}
\times
\underbrace{
\begin{pmatrix}
\cos\theta & -\sin\theta & 0 \\
\sin\theta & \cos\theta & 0 \\
0 & 0 & 1
\end{pmatrix}
}_{(3)\text{绕原点顺时针旋转}\theta \text{弧度}}
\times
\underbrace{
\begin{pmatrix}
-1? & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}
}_{(2)\text{绕Y轴水平翻转}}
\times
\underbrace{
\begin{pmatrix}
1 & 0 & -\frac{\text{width}}{2} \\
0 & 1 & -\frac{\text{height}}{2} \\
0 & 0 & 1
\end{pmatrix}
}_{(1)\text{视觉中心平移至}(0，0)}
$$

### 2.2动画

- 此处矩阵链为标准的语义顺序，实际代码运行多重矩阵运算带来的额外开销需要将部分矩阵合并以避免无效计算

$$ \theta=\left( \texttt{m\_rotationAngle} \times \frac{5294}{360}\right)\times \frac{\pi}{180}$$

$$
\mathbf{T}_{A} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\text{scale}_{x} & 0 & 0 \\
0 & \text{scale}_{y} & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
-1? & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}\\
\times
\begin{pmatrix}
\cos\theta & -\sin\theta & 0 \\
\sin\theta & \cos\theta & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 & \frac{\text{resolution}}{1536}\cdot s & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -97\times k \\
0 & 1 & -97\times k\\
0 & 0 & 1
\end{pmatrix}$$

#### 2.2.1修正列表

- 此表只适用于国际版线性地图
- `worldId≥12`对应`Penny Pursuit`以及`Modern Day`
- 无论是国际版还是中文版迷宫地图时期所有岛屿动画都是 $s = \frac{1536}{1200}, k = 2$

| 条件                                                         | s         | k         |
|------------------------------------------------------------| --------- | --------- |
| $$ worldId=2, animId \in \{9\}$$                           | $$ \frac{1536}{1200} $$ | $$ 2 $$ |
| $$ worldId=3, animId \in \{1,2,3,4,5,6,7,8,9,10,11,12\} $$ | $$ \frac{1536}{1200} $$ | $$ 2 $$  |
| $$ worldId=4, animId \in \{2, 3, 4, 5, 10, 11\} $$         | $$ \frac{1536}{1200} $$ | $$ 2 $$ |
| $$ worldId=5, animId \in \{1, 2, 3, 4, 5\} $$              | $$ \frac{1536}{1200} $$ | $$ 2 $$  |
| $$ worldId=11, animId \in \{28, 29, 30, 31, 32, 33\} $$    | $$ \frac{1536}{1200} $$ | $$ 2 $$  |
| $$ worldId≥12, animId  \in \{3, 6, 7, 8, 9, 11\} $$        | $$ 1 $$ | $$ 2 $$ |
| $$\text{其它情况}$$                                            | $$ 1 $$ | $$ \frac{1536}{600} $$ |

---

## 3. eventList 结点仿射变换

### 常规事件动画修正列表

| eventType | s | k |
|---|---|---|
| level | $$ \frac{1536}{1200} $$ | $$ 2 $$ |
| plantbox、plant                              | $$ \frac{1536}{1200} $$ | $$ 2 $$ |
| star_gate                                    | $$ \frac{1536}{1200} $$ | $$ 2 $$ |
| key_gate                                     | $$ 1 $$                 | $$ \frac{1536}{600} $$ |
| giftbox                                      | $$ \frac{1536}{1200} $$ | $$ 2 $$ |


### 特殊子动画修正列表

| eventType | resource ID | s | k |
|---|---|---|---|
|upgrade| `POPANIM_EFFECTS_COLLECTED_UPGRADE_EFFECT`   | $$ \frac{1536}{1200} $$ | $$ 2 $$ |
|level| `POPANIM_WORLDMAP_ZOMBOSS_NODE_%s`<br>`POPANIM_WORLDMAP_ZOMBOSS_NODE_HOLOGRAM`           | $$ 1 $$ | $$ \frac{1536}{600} $$ |

### 图像修正列表


| eventType |resource ID| s | k |
|---------|----------------------------------------------------------|---|---|
| plant   | `IMAGE_UI_PACKETS_%s`                                                                                  | $$ \frac{1200}{1536} $$ | $$ 2 $$ |
| upgrade | `IMAGE_WORLDMAP_COMMON_UPGRADE_%s`                                                                     | $$ 1 $$                 | $$ \frac{1536}{600} $$ |
| pinata  | `IMAGE_WORLDMAP_SPINE_PINATAS_PINATA_%s_SPINE`<br>`IMAGE_WORLDMAP_SPINE_PINATAS_PINATAS_DUST_SPINE_%s` | $$ 1 $$ | $$ \frac{1536}{600} $$ |

### 3.1 level（包括 minigame、miniboss、dangerroom 等子类，boss 除外）

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 &  \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -98\times k \\
0 & 1 & -104\times k\\
0 & 0 & 1
\end{pmatrix}
$$

#### 3.1.1 dangerroom

图标

resource id：`IMAGE_WORLDMAP_DANGER_LEVEL_%s`

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -2 \cdot \frac{\text{resolution}}{600}\\
0 & 1 & 45 \cdot \frac{\text{resolution}}{600} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -\dfrac{\text{width}}{2} \\
0 & 1 & -\dfrac{\text{height}}{2} \\
0 & 0 & 1
\end{pmatrix}
$$


#### 3.1.2 LevelNodeType == boss

- miniboss 按 level 方式处理

##### POPANIM_WORLDMAP_ZOMBOSS_NODE_%s

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 &  \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -76\times k \\
0 & 1 & -90\times k\\
0 & 0 & 1
\end{pmatrix}
$$

##### POPANIM_WORLDMAP_ZOMBOSS_NODE_HOLOGRAM

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 &  \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -76\times k \\
0 & 1 & -180\times k\\
0 & 0 & 1
\end{pmatrix}
$$

### 3.2 plantbox、plant

#### 动画（plant、sprout 动画）

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 &  \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -98\times k \\
0 & 1 & -115\times k\\
0 & 0 & 1
\end{pmatrix}
$$

#### plantPacket(图像)

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
s & 0 & 0 \\
0 &  s & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -\frac{\text{width}}{2} \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & -\text{height}\cdot0.34 \cdot k \\
0 & 0 & 1
\end{pmatrix}
$$

### 3.3 upgrade

#### 图像(`WorldMapEventStatus==locked`)

$$
\mathbf{T}_{static_u} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -\frac{\text{width}}{2} \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & -\text{height}\cdot0.34\cdot k \\
0 & 0 & 1
\end{pmatrix}
$$

#### POPANIM_EFFECTS_COLLECTED_UPGRADE_EFFECT(`WorldMapEventStatus==cleared`)

##### 一般upgrade物品的背景动画
$$
\mathbf{T} =
{T}_{static_u}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 &  \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -67\times k \\
0 & 1 & -71\times k\\
0 & 0 & 1
\end{pmatrix}
$$

##### diamond的背景动画

$$
\mathbf{T} =
{T}_{static_u}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 &  \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -81\times k \\
0 & 1 & -80\times k\\
0 & 0 & 1
\end{pmatrix}
$$

### 3.4 star_gate

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 &  \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -92\times k \\
0 & 1 & -134\times k\\
0 & 0 & 1
\end{pmatrix}
$$

### 3.5 key_gate

#### key_gate 主体动画
$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 & \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -76\times k \\
0 & 1 & -96\times k\\
0 & 0 & 1
\end{pmatrix}
$$

#### key_gate 旗帜
- key_gate旗帜在特定条件下的原地翻转只影响本身，而其附属元素只依赖其位移并不会跟随翻转，也就是忽略最右侧的平移负宽+翻转矩阵
- keygate_flag的偏移数据是为前三个世界设计的，后续世界的门动画尺寸不一需要单独设计偏移，子元素的偏移数据不用改

| 条件 | `keygate_flag.png` | `info_icon.png` (`cleared`) | `icon_key_%s.png` (`locked`) | 数字 (`locked`) |
|------|-------------------|----------------------------|------------------------------|----------------|
| `key_gate.x - parent.x >= 1` <br>`&&`<br>` m_isArtflip == false` | $$\mathbf{T}_{flag}=\begin{pmatrix} 1 & 0 & X_{w} \\ 0 & 1 & Y_{w} \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & -70 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & -63\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} -1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & -Width \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix} 20\cdot \frac{ 1536}{600\cdot 117} & 0 & 0 \\ 0 & 20\cdot \frac{ 1536}{600\cdot 117} & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 18 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 8\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix}0.5& 0 & 0 \\ 0 &0.5& 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & -\dfrac{Width}{4} \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 16 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 7 \cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix} 1 & 0 & -8 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 4\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ |
| `key_gate.x - parent.x >= 1` <br>`&&`<br>` m_isArtflip == true` | $$\mathbf{T}_{flag}=\begin{pmatrix} 1 & 0 & X_{w} \\ 0 & 1 & Y_{w} \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & -70 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & -41\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} -1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & -Width \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix} 20\cdot \frac{ 1536}{600\cdot 117} & 0 & 0 \\ 0 & 20\cdot \frac{ 1536}{600\cdot 117} & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 18 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 8\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix}0.5& 0 & 0 \\ 0 &0.5& 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & -\dfrac{Width}{4} \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 16 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 7 \cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix} 1 & 0 & -8 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 4\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ |
| `key_gate.x - parent.x < 1` <br>`&&`<br>` m_isArtflip == false` | $$\mathbf{T}_{flag}=\begin{pmatrix} 1 & 0 & X_{w} \\ 0 & 1 & Y_{w} \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 20 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & -41\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix} $$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix} 20\cdot \frac{ 1536}{600\cdot 117} & 0 & 0 \\ 0 & 20\cdot \frac{ 1536}{600\cdot 117} & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 18 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 8\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix}0.5& 0 & 0 \\ 0 &0.5& 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & -\dfrac{Width}{4} \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 20 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 4 \cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix} 1 & 0 & -4 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 5\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ |
| `key_gate.x - parent.x < 1` <br>`&&`<br>` m_isArtflip == true` | $$\mathbf{T}_{flag}=\begin{pmatrix} 1 & 0 & X_{w} \\ 0 & 1 & Y_{w} \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 38 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & -63\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix} $$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix} 20\cdot \frac{ 1536}{600\cdot 117} & 0 & 0 \\ 0 & 20\cdot \frac{ 1536}{600\cdot 117} & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 18 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 8\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix}0.5& 0 & 0 \\ 0 &0.5& 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & -\dfrac{Width}{4} \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \times \begin{pmatrix} 1 & 0 & 20 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 4 \cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ | $$\mathbf{T}=\mathbf{T}_{flag} \times \begin{pmatrix} 1 & 0 & -4 \cdot \frac{\text{resolution}}{600} \\ 0 & 1 & 5\cdot \frac{\text{resolution}}{600} \\ 0 & 0 & 1 \end{pmatrix}$$ |


### 3.6 doodad

#### doodad 图像

$$
\theta=- \texttt{m\_rotationAngle} \times \frac{\pi}{180}
$$

$$
\mathbf{T}_{I} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\text{scale}_{x} & 0 & 0 \\
0 & \text{scale}_{y} & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & -\frac{\text{height}}{2} \\
0 & 0 & 1
\end{pmatrix}\\
\times
\begin{pmatrix}
\cos\theta & -\sin\theta & 0 \\
\sin\theta & \cos\theta & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
-1? & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -\frac{\text{width}}{2} \\
0 & 1 & -\frac{\text{height}}{2} \\
0 & 0 & 1
\end{pmatrix}
$$

#### doodad动画

- k, s 与mapPieces中的岛屿动画完全一致

$$
\theta=\left( \texttt{m\_rotationAngle} \times \frac{5294}{360}\right)\times \frac{\pi}{180}
$$

$$
\mathbf{T}_{A} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\text{scale}_{x} & 0 & 0 \\
0 & \text{scale}_{y} & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
-1? & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}\\
\times
\begin{pmatrix}
\cos\theta & -\sin\theta & 0 \\
\sin\theta & \cos\theta & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 & \frac{\text{resolution}}{1536}\cdot s & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -97\times k \\
0 & 1 & -97\times k\\
0 & 0 & 1
\end{pmatrix}
$$


### 3.7 giftbox

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\frac{\text{resolution}}{1536}\cdot s & 0 & 0 \\
0 & \frac{\text{resolution}}{1536}\cdot s& 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -100\times k \\
0 & 1 & -100\times k\\
0 & 0 & 1
\end{pmatrix}
$$

### 3.8 pinata

$$
\mathbf{T} =
\begin{pmatrix}
1 & 0 & X_{w} \\
0 & 1 & Y_{w} \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -\frac{\text{width}}{2} \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & -\text{height}\cdot0.31 \cdot k \\
0 & 0 & 1
\end{pmatrix}
$$

---

## 4. 迷宫地图独特元素

### 迷宫路径Tile行列映射

$$\begin{pmatrix} \texttt{m\_position.x}\cdot 3 \\ \texttt{m\_position.y} \cdot 3\\ 1 \end{pmatrix} = \begin{pmatrix} 77.74 & -62.35 & 0 \\ 32.90 & 37.10 & 0 \\ 0 & 0 & 1 \end{pmatrix} \begin{pmatrix} \text{col} \cdot 3\\ \text{row} \cdot 3\\ 1 \end{pmatrix} $$

#### 迷宫路径

$$
\text{MapEventItem } A, \text{MapEventItem } B, \text{MapEventItem } C
$$

$$
B\texttt{.m\_parentEvent}==A\texttt{.m\_name}
$$

$$
B\texttt{.m\_unlockedFrom}==C\texttt{.m\_name}
$$

- Tile 全部采用中心对齐坐标绘制，tile实际渲染坐标依然是$(X_w, Y_w)$的插值。
- 路径沿最接近连线方向的主轴逐格步进,不包含起点和终点。
- 每个 Tile 保存四方向（UL、UR、DL、DR）的 in/out 信息。
- 根据 child.m_parentEvent==parent.m_name 建立路径，当结点C的`WorldMapEventStatus`==`cleared`时才会铺设从A到B草坪动画

| 掩码 (mask) | 连接方向| empty纹理索引 | empty纹理文件名 |
|-------------|---------------------------|----------|------------|
| 0 (0000)    | 无                        | 0        | 无纹理（不绘制） |
| 1 (0001)    | ur                        | 2        | empty_ur_dl.png |
| 2 (0010)    | ul                        | 3        | empty_ul_dr.png |
| 3 (0011)    | ur + ul                   | 4        | empty_ul_ur_dr.png |
| 4 (0100)    | dr                        | 3        | empty_ul_dr.png |
| 5 (0101)    | ur + dr                   | 6        | empty_ul_ur_dr.png |
| 6 (0110)    | ul + dr                   | 3        | empty_ul_dr.png |
| 7 (0111)    | ur + ul + dr              | 4        | empty_ul_ur_dr.png |
| 8 (1000)    | dl                        | 2        | empty_ur_dl.png |
| 9 (1001)    | ur + dl                   | 2        | empty_ur_dl.png |
| 10 (1010)   | ul + dl                   | 7        | empty_ul_ur_dl.png |
| 11 (1011)   | ur + ul + dl              | 5        | empty_ul_ur_dl.png |
| 12 (1100)   | dr + dl                   | 6        | empty_ur_dl_dr.png |
| 13 (1101)   | ur + dr + dl              | 6        | empty_ur_dl_dr.png |
| 14 (1110)   | ul + dr + dl              | 7        | empty_ul_dl_dr.png |
| 15 (1111)   | ur + ul + dr + dl         | 5        | empty_ul_dl_dr.png |


### 星星

- 世界锚点跟随level
- 星星数量上限只会是0，1，3，所有`m_isChallengeType`==true的level的星星数量上限恒为1
- 当`packages/worldmaplist`的`LastLevel`变为`cleared`时，所有`m_isChallengeType`==false的level的星星数量上限从0变为3
- 无论上限是多少都只根据`PlayerProfile`中的C数据来从左至右替换图标,已获得星数为n,$2^n-1=C$

$$
\mathbf{T}_{mid} =
\begin{pmatrix} 1 & 0 & X_{w} \\ 0 & 1 & Y_{w} \\ 0 & 0 & 1 \end{pmatrix}
\times
\begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & \frac{\text{resolution}}{600} \cdot 22 \\ 0 & 0 & 1 \end{pmatrix}
\times
\begin{pmatrix}
0.65 & 0 & 0 \\
0 & 0.65 & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -\frac{\text{width}}{2} \\
0 & 1 & -\frac{\text{height}}{2} \\
0 & 0 & 1
\end{pmatrix}
$$

$$
\mathbf{T}_{left} =
\begin{pmatrix} 1 & 0 & X_{w} \\ 0 & 1 & Y_{w} \\ 0 & 0 & 1 \end{pmatrix}
\times
\begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & \frac{\text{resolution}}{600} \cdot 22 \\ 0 & 0 & 1 \end{pmatrix}
\times
\begin{pmatrix}
0.65 & 0 & 0 \\
0 & 0.65 & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -\frac{\text{3 width}}{2} \\
0 & 1 & -\frac{\text{height}}{2} \\
0 & 0 & 1
\end{pmatrix}
$$

$$
\mathbf{T}_{right} =
\begin{pmatrix} 1 & 0 & X_{w} \\ 0 & 1 & Y_{w} \\ 0 & 0 & 1 \end{pmatrix}
\times
\begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & \frac{\text{resolution}}{600} \cdot 22 \\ 0 & 0 & 1 \end{pmatrix}
\times
\begin{pmatrix}
0.65 & 0 & 0 \\
0 & 0.65 & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & +\frac{\text{width}}{2} \\
0 & 1 & -\frac{\text{height}}{2} \\
0 & 0 & 1
\end{pmatrix}
$$

---

## 5. 线性地图独特元素

### 线性地图路径光束动画仿射变换

$$
\text{MapEventItem } A, \text{MapEventItem } B
$$

$$
B\texttt{.m\_parentEvent}==A\texttt{.m\_name}
$$

$$
midX=\frac{X_{wA}+X_{wB}}{2}, \quad
midY=\frac{Y_{wA}+Y_{wB}}{2}
$$

$$
\Delta x = X_{wB}-X_{wA}, \quad
\Delta y = Y_{wB}-Y_{wA}
$$

$$
scaleX=\frac{\sqrt{\Delta x^2+\Delta y^2}}{130\cdot\frac{\text{resolution}}{1536}}
$$

$$
\varphi=\operatorname{atan2f}(\Delta y, \Delta x)
$$

$$
\mathbf{P} =
\begin{pmatrix}%
1 & 0 & midX \\
0 & 1 & midY \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
\cos\varphi & -\sin\varphi & 0 \\
\sin\varphi & \cos\varphi & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
scaleX  & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}%scaleX
\frac{\text{resolution}}{1536}  & 0 & 0 \\
0 & \frac{\text{resolution}}{1536} & 0 \\
0 & 0 & 1
\end{pmatrix}
\times
\begin{pmatrix}
1 & 0 & -97\times 2 \\
0 & 1 & -97\times 2 \\
0 & 0 & 1
\end{pmatrix}
$$

---

## 6. 渲染排序与图层规则

worldmap 的渲染划分为四个独立图层。

其中 **Map Pieces Layer** 保留节点自身 `m_parallaxLayer`(迷宫地图`m_parallaxLayer`恒为0)；其余三大层 **`m_parallaxLayer` 恒为 0**。

整体覆盖关系（下层 → 上层）：

```text
Map Pieces Layer
    ↓
Zomboss Stage Layer
    ↓
Map Path Layer
    ↓
Event Layer
```

### 6.1 Map Pieces Layer

**包含元素**

* `m_mapPieces`

**排序优先级规则**

1. `m_parallaxLayer`（降序）
2. `m_drawLayer`（升序）
3. `m_position.y`（升序）
4. `m_position.x`（升序）



### 6.2 Zomboss Stage Layer（线性地图独有）

**包含元素**

* `POPANIM_WORLDMAP_ZOMBOSS_NODE_%s`

**排序优先级规则**

1. `m_position.y`（升序）
2. `m_position.x`（升序）



### 6.3 Map Path Layer

按最小单元排序。

**排序优先级规则**

1. `m_position.y`（升序）
2. `m_position.x`（升序）


### 6.4 Event Layer

**包含元素**

`m_eventList` 中的所有事件节点。

**排序优先级规则**

1. `m_drawLayer`（升序）
2. `m_position.y`（升序）
3. `m_position.x`（升序）


---


# 部分工具的代码实现

In [ ]:
import math
from typing import List

# Type alias equivalent: SimpleMatrix6 = [a, b, c, d, tx, ty]
# Layout:
# [ a  c  tx ]
# [ b  d  ty ]
# [ 0  0   1 ]

class Matrix:
    
    identity: tuple = (1.0, 0.0, 0.0, 1.0, 0.0, 0.0)

    @staticmethod
    def translate(tx: float, ty: float) -> List[float]:
        return [1.0, 0.0, 0.0, 1.0, tx, ty]

    @staticmethod
    def scale(sx: float, sy: float) -> List[float]:
        return [sx, 0.0, 0.0, sy, 0.0, 0.0]

    @staticmethod
    def rotate(rad: float) -> List[float]:
        cos = math.cos(rad)
        sin = math.sin(rad)
        return [cos, sin, -sin, cos, 0.0, 0.0]

    @staticmethod
    def multiply(a: List[float], b: List[float]) -> List[float]:
        return [
            a[0] * b[0] + a[2] * b[1],
            a[1] * b[0] + a[3] * b[1],
            a[0] * b[2] + a[2] * b[3],
            a[1] * b[2] + a[3] * b[3],
            a[0] * b[4] + a[2] * b[5] + a[4],
            a[1] * b[4] + a[3] * b[5] + a[5],
        ]

    @staticmethod
    def inverse(mat: List[float]) -> List[float]:
        det = mat[0] * mat[3] - mat[1] * mat[2]
        if det == 0:
            return list(Matrix.identity)

        inv = 1.0 / det

        # Inverse of the linear part
        linearInv = [
             mat[3] * inv,
            -mat[1] * inv,
            -mat[2] * inv,
             mat[0] * inv,
            0.0,
            0.0,
        ]

        # M⁻¹ = L⁻¹ · T(-tx, -ty)
        return Matrix.multiply(
            linearInv,
            Matrix.translate(-mat[4], -mat[5])
        )

    @staticmethod
    def transformPoint(mat: List[float], x: float, y: float) -> List[float]:
        m = Matrix.translate(x, y)
        P = Matrix.multiply(mat, m)
        return [P[4], P[5]]


def degreeToRad(angle: float) -> float:
    return -angle * (math.pi / 180.0)


# Column vectors defining the grid basis
col1 = [77.74, 32.9]
col2 = [-62.35, 37.1]


def toColRow(
    X: float,
    Y: float,
    snap: bool
) -> List[float]:
    # Build the transformation matrix from columns
    M = [
        col1[0], col1[1],
        col2[0], col2[1],
        0.0, 0.0,
    ]

    inv = Matrix.inverse(M)
    p = Matrix.transformPoint(inv, X, Y)

    col = round(p[0]) if snap else p[0]
    row = round(p[1]) if snap else p[1]

    return [col, row]


def fromColRow(
    col: float,
    row: float,
) -> List[float]:
    M = [
        col1[0], col1[1],
        col2[0], col2[1],
        0.0, 0.0,
    ]

    return Matrix.transformPoint(M, col, row)